# IsoCourt — full MediaPipe pose cache (no training)

Downloads **FineBadminton-20K** from Hugging Face (via `prepare_finebadminton_20k.py`), then builds **`pose_cache_mediapipe.pt`** for the merged list so every trainer can reuse it.

**Runtime:** CPU is fine (MediaPipe runs on CPU). GPU runtime is optional and does not speed this step much.

Output is saved to **Google Drive** so a Colab disconnect does not lose a multi-hour run.

In [ ]:
import os
print('Working dir will be /content/IsoCourt after clone.')

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/IsoCourt/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)
POSE_OUT = os.path.join(DRIVE_CKPT, 'pose_cache_mediapipe.pt')
print('Pose cache will be written to:', POSE_OUT)

## 2 — Clone repo & install dependencies

In [ ]:
REPO = 'https://github.com/Navneethd8/IsoCourt.git'
BRANCH = 'main'

if not os.path.isdir('/content/IsoCourt'):
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 -b {BRANCH} {REPO} /content/IsoCourt
else:
    !cd /content/IsoCourt && git pull --ff-only

%cd /content/IsoCourt

In [ ]:
!pip install -q opencv-python-headless mediapipe tqdm huggingface_hub torch torchvision

## 3 — Download FineBadminton-20K from Hugging Face & merge labels

Same step as the training notebooks: writes `backend/data/transformed_combined_rounds_output_en_evals_translated.json` and snapshot under `backend/data/FineBadminton-20K/`.

In [ ]:
DATA_DIR = '/content/IsoCourt/backend/data/FineBadminton-20K'
MERGED_JSON = '/content/IsoCourt/backend/data/transformed_combined_rounds_output_en_evals_translated.json'

if not os.path.isfile(MERGED_JSON):
    print('Downloading FineBadminton-20K from HuggingFace Hub...')
    !python backend/pipelines/vlm/common/prepare_finebadminton_20k.py \
        --local-dir {DATA_DIR} --max-workers 8
else:
    print(f'Merged JSON already exists: {MERGED_JSON}')

## 4 — Build full pose cache (long-running)

Uses `build_full_pose_cache.py`. If the file on Drive already exists and **N** matches the dataset, the script **skips** unless you add `--force` to the command below.

After this finishes, point any Colab training notebook at `POSE_OUT` with `--pose-cache-path` / `pose_cache_path=` (same as `Train_CNN_LSTM.ipynb`).

In [ ]:
!python backend/pipelines/training/build_full_pose_cache.py --data-root backend/data --list-file backend/data/transformed_combined_rounds_output_en_evals_translated.json --output {POSE_OUT}

In [ ]:
import os
if os.path.isfile(POSE_OUT):
    sz = os.path.getsize(POSE_OUT) / (1024 ** 3)
    print(f'Pose cache ready: {POSE_OUT} ({sz:.2f} GiB)')
else:
    print('Pose cache not found — check errors in the cell above.')